## Fine tuning of LIama small model

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer


In [2]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer

c:\Users\mikes\Documents\STUDY\LLM-zoomcamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path
import json

In [4]:
# -------- CONFIG --------
PARENT_DIR = Path("C:/Users/mikes/Documents/STUDY/llm_fine_tuning_data")
DATA_FILE = PARENT_DIR / "raw_qa_merged.jsonl"

TRAIN_FILE = PARENT_DIR / "train.jsonl"
VAL_FILE = PARENT_DIR / "val.jsonl"
# ------------------------

In [5]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-6976986b-2438c1a831be130f3d662f74;b12c0ae3-1509-476b-8f65-2a88a3d04820)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
MAX_LEN = 2048

def tokenize_and_mask(example):
    messages = example["messages"]

    # Build text without assistant content (for mask length)
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1] + [{"role": "assistant", "content": ""}],
        tokenize=False,
        add_generation_prompt=False,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=True).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=True, truncation=True, max_length=MAX_LEN).input_ids

    labels = full_ids.copy()
    # Mask system/user and any pre-answer tokens
    mask_len = min(len(prompt_ids), len(full_ids))
    for i in range(mask_len):
        labels[i] = -100

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }



In [ ]:
# Load your train.jsonl produced above
train_ds = load_dataset("json", data_files=str(TRAIN_FILE))["train"]
train_tok = train_ds.map(tokenize_and_mask, remove_columns=train_ds.column_names)

# Pad in the collator and keep -100 on padding in labels
class DataCollatorForAnswerMask:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids = [torch.tensor(x["input_ids"]) for x in batch]
        attention_mask = [torch.tensor(x["attention_mask"]) for x in batch]
        labels = [torch.tensor(x["labels"]) for x in batch]

        padded = self.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True,
            return_tensors="pt",
        )

        # Pad labels to match input_ids, using -100 as ignore_index
        max_len = padded["input_ids"].shape[1]
        padded_labels = torch.full((len(labels), max_len), -100, dtype=torch.long)
        for i, lab in enumerate(labels):
            padded_labels[i, : lab.shape[0]] = lab

        padded["labels"] = padded_labels
        return padded

data_collator = DataCollatorForAnswerMask(tokenizer)

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl"
    }
)

def format_chat(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False
    )

dataset = dataset.map(lambda x: {"text": format_chat(x)})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./lora-ml-qa",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048
)

trainer.train()
